In [1]:
import pandas as pd
from pymongo import MongoClient
from joblib import Parallel, delayed
from tqdm import tqdm

client = MongoClient("127.0.0.1", 27017)
db = client["bridge"]
py_nonfixed_bumping_col = db["py_nonfixed_version_bumping_commits"]
java_nonfixed_bumping_col = db["java_nonfixed_version_bumping_commits"]

pipeline = [{"$group": {"_id": "$commit"}}]
num_java_nonfixed_bumping = len(list(java_nonfixed_bumping_col.aggregate(pipeline)))
num_py_nonfixed_bumping = len(list(py_nonfixed_bumping_col.aggregate(pipeline)))
print(num_java_nonfixed_bumping, "nonfixed Java version bumping commits")
print(num_py_nonfixed_bumping, "nonfixed Python version bumping commits")

3054 nonfixed Java version bumping commits
2110515 nonfixed Python version bumping commits


In [ ]:
import re
from poetry.core.constraints.version.parser import parse_constraint


def maven2py(spec: str):
    spec = re.sub(r"\s+", "", spec)
    pattern = r"(?<=[\])]),(?=[\[\(])"
    res = []
    for s in re.split(pattern, spec):
        tmp = []
        if "," not in s:
            res.append("==" + s.strip("[]"))
            continue
        l, r = s.split(",", 1)
        if l == "[":
            continue
        if r == "]":
            continue
        if l != "(":
            if l[0] == "(":
                tmp.append(f">{l[1:]}")
            else:
                tmp.append(f">={l[1:]}")
        if r != ")":
            if r[-1] == ")":
                tmp.append(f"<{r[:-1]}")
            else:
                tmp.append(f"<={r[:-1]}")
        res.append(",".join(tmp))
    return res


def constraint_overlap(v1, v2):
    if (v1 == "") or (v2 == ""):
        return True
    try:
        v1 = parse_constraint(v1)
        v2 = parse_constraint(v2)
        if v1.intersect(v2).is_empty():
            return False
    except:
        return True
    return True


def has_overlap_java(doc):
    v1 = doc["version_before"]
    v2 = doc["version_after"]
    specs1 = maven2py(v1)
    specs2 = maven2py(v2)
    for s1 in specs1:
        for s2 in specs2:
            if constraint_overlap(s1, s2):
                return None
    return doc


def has_overlap_py(doc):
    v1 = doc["version_before"]
    v2 = doc["version_after"]
    if constraint_overlap(v1, v2):
        return None
    return doc

In [ ]:
java_results = Parallel(n_jobs=100)(
    delayed(has_overlap_java)(doc)
    for doc in tqdm(
        java_nonfixed_bumping_col.find({}),
        total=java_nonfixed_bumping_col.estimated_document_count(),
    )
)
py_results = Parallel(n_jobs=100)(
    delayed(has_overlap_py)(doc)
    for doc in tqdm(
        py_nonfixed_bumping_col.find({}, projection),
        total=py_nonfixed_bumping_col.estimated_document_count(),
    )
)

100%|██████████| 5174367/5174367 [02:29<00:00, 34597.18it/s]


In [ ]:
java_nonoverlap_commits = pd.DataFrame([doc for doc in java_results if doc]).drop(
    columns=["_id"]
)
py_nonoverlap_commits = pd.DataFrame([doc for doc in py_results if doc]).drop(
    columns=["_id"]
)
len(java_nonoverlap_commits), len(py_nonoverlap_commits)

(1831, 562540)

In [ ]:
db["java_nonoverlap_commits"].insert_many(java_nonoverlap_commits.to_dict("records"))
db["py_nonoverlap_commits"].insert_many(py_nonoverlap_commits.to_dict("records"))

In [ ]:
def merge_cfg_files(df):
    cfg_file_changes = {}
    for _, doc in df.iterrows():
        filepath = doc["filepath"]
        cfg_file_changes.setdefault(filepath, [])
        cfg_file_changes[filepath].append(
            {
                "package": doc["package"],
                "version_before": doc["version_before"],
                "version_after": doc["version_after"],
            }
        )
    cfg_file_changes = [
        {"filepath": k, "dependency_changes": v} for k, v in cfg_file_changes.items()
    ]
    return cfg_file_changes


java_cfg_changes = (
    java_nonoverlap_commits.groupby("commit").apply(merge_cfg_files).to_dict()
)
py_cfg_changes = (
    py_nonoverlap_commits.groupby("commit").apply(merge_cfg_files).to_dict()
)

/tmp/ipykernel_1101570/2010731892.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  java_cfg_changes = java_nonoverlap_commits.groupby("commit").apply(merge_cfg_files)
/tmp/ipykernel_1101570/2010731892.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  py_cfg_changes = py_nonoverlap_commits.groupby("commit").apply(merge_cfg_files)


In [59]:
print(len(java_cfg_changes), "Java nonfixed version contraint nonoveralpped commits")
print(len(py_cfg_changes), "Python nonfixed version contraint nonoveralpped commits")

984 Java nonfixed version contraint nonoveralpped commits
363326 Python nonfixed version contraint nonoveralpped commits


In [ ]:
from woc.local import WocMapsLocal
from tqdm.notebook import tqdm

woc = WocMapsLocal()


def file_filter_py(filepath: str):
    parts = filepath.split("/")
    if parts[-1] == "setup.py":
        return True
    if "site-packages" in parts:
        return True
    return False


def get_code_file_changes(commit: str, lang: str, file_filter=None):
    code_file_changes = []

    try:
        fbbs = woc.get_values("c2fbb", commit)
    except:
        return []
    for f, nb, ob in fbbs:
        if not f.endswith(lang):
            continue
        if file_filter and file_filter(f):
            continue
        if len(nb) != 40:
            continue
        if len(ob) != 40:
            continue
        code_file_changes.append({"filepath": f, "new_blob": nb, "old_blob": ob})
    return code_file_changes


def get_update_commits():
    java_data = []
    for commit, cfg_file_changes in tqdm(java_cfg_changes.items()):
        code_file_changes = get_code_file_changes(commit, "java")
        if not code_file_changes:
            continue
        java_data.append(
            {
                "commit": commit,
                "configuration_files": cfg_file_changes,
                "code_files": code_file_changes,
            }
        )
    print(len(java_data), "nonfixed Java update commits")
    db["java_nonfixed_update_commits"].drop()
    db["java_nonfixed_update_commits"].insert_many(java_data)
    db["java_nonfixed_update_commits"].create_index("commit")

    py_data = []
    for commit, cfg_file_changes in tqdm(py_cfg_changes.items()):
        code_file_changes = get_code_file_changes(commit, "py", file_filter_py)
        if not code_file_changes:
            continue
        py_data.append(
            {
                "commit": commit,
                "configuration_files": cfg_file_changes,
                "code_files": code_file_changes,
            }
        )
    print(len(py_data), "nonfixed Python update commits")
    db["py_nonfixed_update_commits"].drop()
    db["py_nonfixed_update_commits"].insert_many(py_data)
    db["py_nonfixed_update_commits"].create_index("commit")


get_update_commits()

  0%|          | 0/984 [00:00<?, ?it/s]

539 nonfixed Java update commits


  0%|          | 0/363326 [00:00<?, ?it/s]

156500 nonfixed Python update commits
